<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_11_2_critical_errors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 11: Troubleshooting and Evaluating PyTorch Models**  

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 11 Material

* Part 11.1: Debugging with PyTorch Networks [[Notebook]](t81_558_class_11_1_debugging_pytorch.ipynb)
* **Part 11.2: Critical PyTorch Errors** [[Notebook]](t81_558_class_11_2_critical_errors.ipynb)
* Part 11.3: Overfitting and Underfitting [[Notebook]](t81_558_class_11_3_overfitting.ipynb)
* Part 11.4: Vanishing and Exploding Gradients [[Notebook]](t81_558_class_11_4_gradients.ipynb)
* Part 11.5: Error Metrics Beyond Accuracy [[Notebook]](t81_558_class_11_5_metrics.ipynb)

# Google CoLab Instructions

The following code checks that Google CoLab is running and sets up the correct hardware settings for PyTorch.

In [1]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: not using Google CoLab


Using device: mps


# Part 11.2: Critical PyTorch Errors

The previous section dealt with silent failures, where the code runs but the model does not learn. This section covers the opposite: the loud failures, where PyTorch raises an exception and stops. These are, paradoxically, the friendlier kind of bug, because the interpreter tells you something is wrong instead of leaving you to discover it later. The difficulty is that PyTorch's error messages, while precise, are written in the vocabulary of tensor operations and can be intimidating until you learn to read them.

A handful of errors account for the large majority of what you will encounter in practice. They fall into a few families: shape mismatches, where two tensors have incompatible dimensions; device mismatches, where tensors live on different hardware; dtype mismatches, where a tensor has the wrong numeric type for an operation; and autograd errors, where something about the gradient computation is inconsistent. Learning to recognize each family on sight turns a frustrating hunt into a quick fix.

In each example below we deliberately trigger the error, catch it, and print the message so the notebook keeps running. Then we show the fix. Reproducing an error on purpose, in the smallest possible piece of code, is itself one of the most useful debugging skills: an error you can reproduce in three lines is an error you can understand.

## Shape Mismatch in a Linear Layer

The most common exception in PyTorch is a shape mismatch during matrix multiplication. An `nn.Linear(in_features, out_features)` layer performs a matrix multiply that requires the last dimension of its input to equal `in_features`. If it does not, PyTorch reports that the two matrices cannot be multiplied and, helpfully, prints both shapes.

Read the message literally. It tells you the shapes of the two operands it tried to multiply. Comparing them shows which dimension is wrong, and comparing that against your layer definition shows why.

In [2]:
import torch
import torch.nn as nn

# A Linear(10, 5) expects each input row to have exactly 10 values.
layer = nn.Linear(10, 5)
wrong_input = torch.randn(4, 8)   # 8 features, not 10

try:
    layer(wrong_input)
except RuntimeError as e:
    print("RuntimeError:", e)

# THE FIX: make the input's last dimension match in_features (10).
right_input = torch.randn(4, 10)
print("\nFixed output shape:", tuple(layer(right_input).shape))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x8 and 10x5)

Fixed output shape: (4, 5)


The message names the two shapes it could not multiply: the `(4, 8)` input and the layer's `(10, 5)` weight. Because 8 does not equal 10, the multiply is impossible. The fix is to ensure the data entering the layer has the number of features the layer was defined to accept. When this error appears deep in a network, the forward hooks from the previous section are the fastest way to find which layer the mismatched tensor reached.

## Device Mismatch

PyTorch tensors live on a specific device: the CPU, a CUDA GPU, or Apple's MPS backend. An operation that combines two tensors requires them to be on the same device. The most common version of this bug is a model that has been moved to the GPU while a batch of input data is still on the CPU. PyTorch refuses to run the operation and reports that it expected all tensors to be on the same device.

The fix is to move the input to the model's device with `.to(device)`. The standard pattern, which you have seen throughout this course, is to define a single `device` variable at the top of the notebook and call `.to(device)` on the model once and on every batch as it is produced.

In [3]:
# The model's parameters and its input must be on the same device.
model = nn.Linear(4, 2).to(device)
cpu_input = torch.randn(3, 4)      # created on the CPU by default

try:
    # If `device` is not "cpu", the model is elsewhere and this mismatches.
    output = model(cpu_input)
    print(f"No mismatch on this machine because device == '{device}'.")
    print("Output shape:", tuple(output.shape))
except RuntimeError as e:
    print("RuntimeError:", str(e).splitlines()[0])

# THE FIX: move the input to the same device as the model.
output = model(cpu_input.to(device))
print("Fixed output shape:", tuple(output.shape))

RuntimeError: Tensor for argument input is on cpu but expected on mps
Fixed output shape: (3, 2)


If this notebook runs on a GPU or on Apple's MPS backend, the first call raises a device-mismatch error because the model is on the accelerator and the input is on the CPU. If it runs on a CPU-only machine, there is no second device to mismatch and the call simply succeeds; the printed note reflects which case you are in. Either way, the reliable habit is the same: send every batch to the model's device before the forward pass.

## Dtype Mismatch in the Loss Function

Numeric type errors are subtle because most tensors default to 32-bit floats and "just work" together, so the one case that does not can be surprising. The classic example involves `nn.CrossEntropyLoss`. It expects raw logits as floats, but its *target* must be a `long` (64-bit integer) tensor of class indices. If the class indices are accidentally left as floats, PyTorch reports that it expected a `Long` but found a `Float`.

The message points directly at the fix. When the target holds class indices, convert it to integers with `.long()`. This is one of the most frequent errors for newcomers, precisely because integer labels loaded from a file or computed from a threshold can easily end up as floats.

In [4]:
logits = torch.randn(6, 3)   # 6 samples, 3 classes, raw float logits
loss_fn = nn.CrossEntropyLoss()

# Class-index targets must be integers (Long). Here they were left as floats.
float_targets = torch.tensor([0, 2, 1, 1, 0, 2], dtype=torch.float)

try:
    loss_fn(logits, float_targets)
except (RuntimeError, ValueError) as e:
    print(type(e).__name__ + ":", str(e).splitlines()[0])

# THE FIX: class indices must be a LongTensor.
loss = loss_fn(logits, float_targets.long())
print("\nFixed loss:", round(loss.item(), 4))

RuntimeError: expected target dtype to be Long or Byte, but got Float

Fixed loss: 1.6054


The error names the mismatch precisely: a `Long` was expected but a `Float` was found. Calling `.long()` on the target converts the class indices to the integer type the loss requires. Note the related distinction: `CrossEntropyLoss` expects class *indices* shaped `(batch,)`, not one-hot vectors shaped `(batch, num_classes)`. Passing one-hot targets is a different bug that produces a shape or value error, and the fix there is to convert one-hot encodings back to indices with `argmax`.

## Forgetting zero_grad: A Silent Loud Error

The next error raises no exception at all, which is what makes it dangerous. By default, PyTorch *accumulates* gradients: each call to `backward()` adds the newly computed gradients to whatever is already stored in the `.grad` attributes. This design is deliberate and useful for some techniques, but it means that in an ordinary training loop you must reset the gradients to zero every step with `optimizer.zero_grad()`. Forget it, and the gradients from every past step pile up, producing corrupted updates and unstable training.

Because nothing crashes, the only symptom is a model that trains poorly for no obvious reason. The way to *see* the bug is to watch the gradient norm across steps. Without `zero_grad`, it grows every step as old gradients accumulate; with it, the gradient reflects only the current batch.

In [5]:
model = nn.Linear(4, 1)
x = torch.randn(8, 4)
y = torch.randn(8, 1)
loss_fn = nn.MSELoss()

# WITHOUT zero_grad: gradients accumulate across steps and the norm grows.
print("WITHOUT zero_grad (gradients accumulate):")
for step in range(4):
    loss = loss_fn(model(x), y)
    loss.backward()
    print(f"  step {step}: grad norm = {model.weight.grad.norm():.4f}")

# WITH zero_grad: each step's gradient reflects only the current batch.
print("\nWITH zero_grad (correct):")
for step in range(4):
    model.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    print(f"  step {step}: grad norm = {model.weight.grad.norm():.4f}")

WITHOUT zero_grad (gradients accumulate):
  step 0: grad norm = 1.7333
  step 1: grad norm = 3.4666
  step 2: grad norm = 5.2000
  step 3: grad norm = 6.9333

WITH zero_grad (correct):
  step 0: grad norm = 1.7333
  step 1: grad norm = 1.7333
  step 2: grad norm = 1.7333
  step 3: grad norm = 1.7333


In the first loop the gradient norm climbs steadily because each backward pass adds to the previous gradients. In the second loop, zeroing the gradients first keeps the norm stable and correct across every step. This is why the canonical PyTorch training loop always follows the order: `zero_grad()`, forward, compute loss, `backward()`, `step()`. Committing that sequence to memory prevents one of the most common and hardest-to-spot training bugs.

## Backpropagating Through a Freed Graph

The last error in this section comes from autograd's memory management. To save memory, PyTorch frees the computation graph as soon as `backward()` has used it. Calling `backward()` a second time on the same graph therefore fails, because the intermediate values it needs have already been released. PyTorch reports that you are trying to backpropagate through the graph a second time.

This surprises people who compute two losses from the same forward pass and call `backward()` on each. The usual fix is to combine the losses into a single quantity and call `backward()` once. In the rarer case where two separate backward passes are genuinely required, the first call must pass `retain_graph=True` to keep the graph alive.

In [6]:
x = torch.randn(5, 3, requires_grad=True)
y = (x ** 2).sum()

y.backward()          # first backward: computes gradients, then frees the graph
try:
    y.backward()      # second backward on the same, now-freed graph
except RuntimeError as e:
    print("RuntimeError:", str(e).splitlines()[0])

# THE FIX (when a second backward is truly needed): retain the graph.
z = (x ** 2).sum()
z.backward(retain_graph=True)
z.backward()          # now this works because the graph was kept alive
print("\nSecond backward succeeded after retain_graph=True")

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

Second backward succeeded after retain_graph=True


The first attempt fails because the graph built by the original forward computation was freed after the initial `backward()`. Rebuilding the computation and passing `retain_graph=True` on the first call keeps the intermediate values available for a second pass. In everyday training you rarely need this; if you find yourself reaching for `retain_graph=True`, it is worth first checking whether you can simply sum your losses and call `backward()` once.

These five errors, shape, device, dtype, missing `zero_grad`, and freed-graph, cover the great majority of exceptions you will meet while building PyTorch models. The common thread is that every message is specific: it names the shapes, the devices, or the types involved. Reading the message literally, and reproducing the failure in the smallest possible snippet, turns each one into a routine fix. The next section, [Overfitting and Underfitting](t81_558_class_11_3_overfitting.ipynb), returns to a subtler kind of problem, one that no exception will ever warn you about.